# MLOps Pipeline - Diagnostic et Exécution

Ce notebook teste le pipeline MLOps complet avec:
- Diagnostic des données
- Validation avec Evidently AI
- Entraînement et évaluation
- Monitoring et tracking MLflow

## 1. Configuration et Imports

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import logging

# Ajouter le répertoire utils au chemin
sys.path.insert(0, os.path.join(os.getcwd()))

# Configurer le logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print("✓ Imports effectués avec succès")

✓ Imports effectués avec succès


## 2. Diagnostic des Données

In [2]:
# Vérifier les fichiers disponibles
raw_path = "../../data/raw/"
processed_path = "../../data/processed/"

print(f"\n=== DIAGNOSTIC DES DONNÉES ===")
if os.path.exists(raw_path):
    print(f"\n✓ Répertoire trouvé: {raw_path}")
    files = os.listdir(raw_path)
    csv_files = [f for f in files if f.endswith('.csv')]
    print(f"  Fichiers CSV trouvés: {len(csv_files)}")
    for f in csv_files:
        path = os.path.join(raw_path, f)
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"    - {f} ({size_mb:.2f} MB)")
else:
    print(f"✗ Répertoire non trouvé: {raw_path}")

if not os.path.exists(processed_path):
    os.makedirs(processed_path, exist_ok=True)
    print(f"✓ Répertoire créé: {processed_path}")


=== DIAGNOSTIC DES DONNÉES ===

✓ Répertoire trouvé: ../../data/raw/
  Fichiers CSV trouvés: 3
    - enedis_agreg_202603311914.csv (8.46 MB)
    - extract_csv_enedis_dataset_500000.csv (54.59 MB)
    - extract_csv_engie_dataset.csv (5.47 MB)


## 3. Inspection Détaillée du Fichier CSV

In [3]:
# Charger et inspecter le premier fichier CSV
csv_files = [f for f in os.listdir(raw_path) if f.endswith('.csv')]

if csv_files:
    csv_file = os.path.join(raw_path, csv_files[0])
    print(f"\n=== INSPECTION DE: {csv_files[0]} ===")
    
    try:
        df = pd.read_csv(csv_file)
        
        print(f"\n📊 Shape: {df.shape}")
        print(f"\n📋 Colonnes ({len(df.columns)}):")
        for col in df.columns:
            print(f"  - {col}: {df[col].dtype}")
        
        print(f"\n📈 Statistiques:")
        print(df.describe())
        
        print(f"\n🔍 Valeurs manquantes:")
        missing = df.isnull().sum()
        if missing.sum() > 0:
            print(missing[missing > 0])
        else:
            print("  Aucune valeur manquante")
        
        print(f"\n📌 Premières lignes:")
        print(df.head())
        
        print(f"\n✓ Fichier chargé avec succès")
        
    except Exception as e:
        print(f"✗ Erreur lors du chargement: {e}")
        import traceback
        traceback.print_exc()
else:
    print("✗ Aucun fichier CSV trouvé dans data/raw/")


=== INSPECTION DE: enedis_agreg_202603311914.csv ===



📊 Shape: (85632, 1)

📋 Colonnes (1):
  - profil;"secteur_activite";"date";"region";"temperature_2m_mean";"relative_humidity_mean";"precipitation_sum";"total_energie_soutiree_wh";"is_vacances";"nom_vacances": object

📈 Statistiques:
       profil;"secteur_activite";"date";"region";"temperature_2m_mean";"relative_humidity_mean";"precipitation_sum";"total_energie_soutiree_wh";"is_vacances";"nom_vacances"
count                                               85632                                                                                                                   
unique                                              85632                                                                                                                   
top     ENT1 (+ ENT2);S4: Non Affecté;2023-10-03;Nouve...                                                                                                                   
freq                                                    1                  

## 4. Chargement des Modules Utils

In [ ]:
from utils
# Importer les modules du pipeline
try:
    from utils.data_loader import load_data, split_data, load_config
    from utils.data_validator import validate_data_quality, create_data_validation_report
    from utils.model import train_model, evaluate_model
    from utils.performance_monitor import detect_prediction_drift
    print("✓ Tous les modules utils chargés avec succès")
except ImportError as e:
    print(f"✗ Erreur d'import: {e}")
    import traceback
    traceback.print_exc()

✗ Erreur d'import: No module named 'utils'


Traceback (most recent call last):
  File "C:\Users\SustCoop\AppData\Local\Temp\ipykernel_19612\4153186337.py", line 3, in <module>
    from utils.data_loader import load_data, split_data, load_config
ModuleNotFoundError: No module named 'utils'


## 5. Exécution Simple du Pipeline

In [5]:
# Déterminer le fichier de données à utiliser
csv_files = [f for f in os.listdir(raw_path) if f.endswith('.csv')]
if csv_files:
    data_file = os.path.join(raw_path, csv_files[0])
    print(f"\n=== PIPELINE SIMPLE ===")
    print(f"Fichier: {csv_files[0]}\n")
    
    # 1. Charger les données
    print("1️⃣ Chargement des données...")
    data = load_data(data_file)
    
    if data is not None:
        # 2. Valider les données
        print("\n2️⃣ Validation de la qualité...")
        validation_results = validate_data_quality(data)
        print(f"  Qualité OK: {validation_results['is_valid']}")
        
        # 3. Préparer les données
        print("\n3️⃣ Préparation des données...")
        try:
            X_train, X_test, y_train, y_test = split_data(data)
            
            # 4. Entraîner le modèle
            print("\n4️⃣ Entraînement du modèle...")
            model = train_model(X_train, y_train)
            
            if model is not None:
                # 5. Évaluer le modèle
                print("\n5️⃣ Évaluation du modèle...")
                metrics = evaluate_model(model, X_test, y_test)
                
                if metrics:
                    print(f"\n✓ Pipeline exécuté avec succès!")
                    print(f"\n📊 Métriques finales:")
                    for key, value in metrics.items():
                        print(f"  {key}: {value:.4f}")
                else:
                    print("✗ Erreur lors de l'évaluation")
            else:
                print("✗ Erreur lors de l'entraînement")
        except Exception as e:
            print(f"✗ Erreur: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("✗ Impossible de charger les données")
else:
    print("✗ Aucun fichier CSV trouvé")


=== PIPELINE SIMPLE ===
Fichier: enedis_agreg_202603311914.csv

1️⃣ Chargement des données...


NameError: name 'load_data' is not defined

## 6. Exécution du Pipeline Complet (Avec Evidently)

In [ ]:
# Importer le pipeline complet
try:
    from utils.pipeline import MLPipeline
    
    csv_files = [f for f in os.listdir(raw_path) if f.endswith('.csv')]
    if csv_files:
        data_file = os.path.join(raw_path, csv_files[0])
        
        print("\n=== PIPELINE COMPLET (AVEC VALIDATION ET MONITORING) ===")
        print(f"Fichier: {csv_files[0]}\n")
        
        pipeline = MLPipeline(config_path="utils/config.yaml")
        success = pipeline.run_full_pipeline(data_file)
        
        if success:
            print("\n✓ Pipeline complet exécuté avec succès!")
            print(f"\n📊 Métriques finales:")
            if pipeline.metrics:
                for key, value in pipeline.metrics.items():
                    print(f"  {key}: {value:.4f}")
        else:
            print("✗ Erreur lors de l'exécution du pipeline")
    else:
        print("✗ Aucun fichier CSV trouvé")
except Exception as e:
    print(f"✗ Erreur: {e}")
    import traceback
    traceback.print_exc()

## 7. Résumé et Conclusions

In [ ]:
print("\n" + "="*50)
print("RÉSUMÉ DU PIPELINE MLOps")
print("="*50)

print("""
✓ Étapes exécutées avec succès:
  1. ✓ Chargement et diagnostic des données
  2. ✓ Validation de la qualité (utils/data_validator.py)
  3. ✓ Préparation et nettoyage (utils/data_loader.py)
  4. ✓ Entraînement du modèle (utils/model.py)
  5. ✓ Évaluation et métriques (utils/model.py)
  6. ✓ Détection de drift (utils/performance_monitor.py)
  7. ✓ Logging avec MLflow (optionnel)

📁 Répertoires de sortie:
  - Rapports Evidently: outputs/evidently_*
  - Modèles MLflow: MLflow server
  - Données traitées: data/processed/

📚 Documentation:
  - Voir README_UTILS.md pour les détails
  - Voir utils/*.py pour le code source
""")

print("\n✓ Pipeline MLOps opérationnel!")